# Gradient Flow Analysis

Analyze gradient magnitudes and directions through layers: norm tracking,
component attribution, saturation detection, bottleneck identification, and summary.

## Why This Matters

Gradient flow traces how training signals and attribution scores flow through the network. Understanding gradient structure reveals which components are most trainable, which carry the strongest attribution signals, and how LayerNorm affects learning.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.gradient_flow_analysis import (
    gradient_norm_by_layer, gradient_component_attribution,
    gradient_saturation_analysis, gradient_bottleneck_detection,
    gradient_flow_summary,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Gradient Norms by Layer

Track activation norms and update magnitudes through the network.

In [ ]:
result = gradient_norm_by_layer(model, tokens)
print(f"Growth ratio: {result['mean_growth_ratio']:.4f}")
print(f"Stable: {result['is_stable']}\n")
for p in result['per_layer']:
    bar = '█' * int(min(p['activation_norm'] * 5, 30))
    print(f"  Layer {p['layer']}: act_norm={p['activation_norm']:.4f}, "
          f"update={p['update_norm']:.4f}, ratio={p['update_ratio']:.4f} {bar}")

## Component Attribution

Which component (attention vs MLP) contributes more at each layer?

In [ ]:
result = gradient_component_attribution(model, tokens)
print(f"Dominant component: {result['dominant_component']}\n")
for p in result['per_layer']:
    attn_bar = '█' * int(p['attn_fraction'] * 30)
    mlp_bar = '░' * int(p['mlp_fraction'] * 30)
    print(f"  Layer {p['layer']}: attn={p['attn_fraction']:.2%} {attn_bar} | "
          f"mlp={p['mlp_fraction']:.2%} {mlp_bar}")

## Saturation Analysis

Detect saturated attention (near 0/1) and dead MLP neurons.

In [ ]:
result = gradient_saturation_analysis(model, tokens)
print(f"Any saturated: {result['any_saturated']}\n")
for p in result['per_layer']:
    flag = ' [SATURATED]' if p['is_saturated'] else ''
    print(f"  Layer {p['layer']}: attn_sat={p['attention_saturation']:.4f}, "
          f"mlp_dead={p['mlp_near_zero_fraction']:.4f}{flag}")

## Bottleneck Detection

Identify layers where information flow narrows (low effective rank).

In [ ]:
result = gradient_bottleneck_detection(model, tokens)
print(f"Worst bottleneck: Layer {result['worst_bottleneck_layer']}")
print(f"Any bottleneck: {result['any_bottleneck']}\n")
for p in result['per_layer']:
    flag = ' [BOTTLENECK]' if p['is_bottleneck'] else ''
    print(f"  Layer {p['layer']}: rank={p['update_effective_rank']:.2f}, "
          f"top_sv={p['top_sv_fraction']:.4f}{flag}")

## Flow Summary

Cross-layer summary of gradient flow health.

In [ ]:
result = gradient_flow_summary(model, tokens)
print(f"Stable: {result['is_stable']}")
print(f"Dominant: {result['dominant_component']}")
print(f"Any saturated: {result['any_saturated']}")
print(f"Any bottleneck: {result['any_bottleneck']}\n")
for p in result['per_layer']:
    print(f"  Layer {p['layer']}: norm={p['activation_norm']:.4f}, "
          f"update_ratio={p['update_ratio']:.4f}, "
          f"attn={p['attn_fraction']:.2%}, sat={p['attention_saturation']:.4f}, "
          f"rank={p['effective_rank']:.2f}")